# 02. Tiền xử lý và chia dữ liệu

Notebook này dùng để chuẩn bị dữ liệu trước khi xây dựng mô hình. Sau khi gộp và làm sạch dữ liệu, nhóm chia dữ liệu thành ba tập train, validation và test. Dữ liệu gốc trong `data/raw/` được giữ nguyên để có thể sử dụng lại khi cần.

Kết quả sau khi chạy notebook sẽ tạo ra các file `merged.csv`, `cleaned.csv`, `train.csv`, `valid.csv` và `test.csv`. Đây là các file sẽ được sử dụng trong những notebook tiếp theo.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'data').exists() and (folder / 'src').exists():
            return folder
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.load_data import save_merged_data
from src.data.clean_data import save_cleaned_data
from src.data.split_data import save_splits

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\NhatThien\Documents\GitHub\Nhom9-De18-student-performance-ml


In [ ]:
# Bước 1: gộp hai file raw thành một file trung gian.
merged_df = save_merged_data()
print(f'merged.csv: {merged_df.shape[0]} dòng x {merged_df.shape[1]} cột')
display(merged_df.head(3))

# Bước 2: chuẩn hóa kiểu dữ liệu, kiểm tra giá trị và loại dòng trùng.
cleaned_df, duplicate_count = save_cleaned_data()
print(f'cleaned.csv: {cleaned_df.shape[0]} dòng x {cleaned_df.shape[1]} cột')
print(f'Số dòng trùng đã loại: {duplicate_count}')

# Bước 3: chia train/validation/test theo nhóm học sinh.
train_df, valid_df, test_df = save_splits()
split_sizes = pd.Series({
    'train': len(train_df),
    'valid': len(valid_df),
    'test': len(test_df)
})
display(split_sizes.to_frame('rows'))

merged.csv: 1044 dòng x 34 cột


,subject,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,math,GP,F,18,U,GT3,A,4,4,at_home,...,4,3,4,1,1,3,6,5,6,6
1,math,GP,F,17,U,GT3,T,1,1,at_home,...,5,3,3,1,1,3,4,5,5,6
2,math,GP,F,15,U,LE3,T,1,1,at_home,...,4,3,2,2,3,3,10,7,8,10


cleaned.csv: 1044 dòng x 34 cột
Số dòng trùng đã loại: 0


,rows
train,730
valid,157
test,157


In [ ]:
# Kiểm tra schema và chất lượng sau tiền xử lý.
expected_first_columns = ['subject', 'school', 'sex', 'age', 'address']
assert list(cleaned_df.columns[:5]) == expected_first_columns
assert 'G3' in cleaned_df.columns
assert cleaned_df.isna().sum().sum() == 0
assert len(train_df) + len(valid_df) + len(test_df) == len(cleaned_df)

numeric_columns = ['age', 'Medu', 'Fedu', 'G1', 'G2', 'G3', 'absences']
print('Kiểu dữ liệu các cột số:')
display(cleaned_df[numeric_columns].dtypes.to_frame('dtype'))
print("Phân phối subject trong từng tập:")

for name, frame in {
    "train": train_df,
    "valid": valid_df,
    "test": test_df,
}.items():

    distribution = (
        frame["subject"]
        .value_counts()
        .sort_index()
        .rename("rows")
    )

    print(f"\n{name}:")
    display(distribution.to_frame())

Kiểu dữ liệu các cột số:


,dtype
age,int64
Medu,int64
Fedu,int64
G1,int64
G2,int64
G3,int64
absences,int64


Phân phối subject và nhóm đạt/trượt trong từng tập:

train:


rows
subject    passed      
math       False     91
           True     185
portuguese False     70
           True     384


valid:


rows
subject    passed      
math       False     19
           True      40
portuguese False     15
           True      83


test:


rows
subject    passed      
math       False     20
           True      40
portuguese False     15
           True      82

In [ ]:
from src.data.split_data import (
    create_student_group_id,
    check_no_group_overlap,
)

train_ids = create_student_group_id(train_df)
valid_ids = create_student_group_id(valid_df)
test_ids = create_student_group_id(test_df)

check_no_group_overlap(
    train_ids,
    valid_ids,
    test_ids,
)

overlap_counts = {
    "train_valid": len(set(train_ids) & set(valid_ids)),
    "train_test": len(set(train_ids) & set(test_ids)),
    "valid_test": len(set(valid_ids) & set(test_ids)),
}

display(
    pd.Series(
        overlap_counts,
        name="overlap_students",
    ).to_frame()
)

assert all(value == 0 for value in overlap_counts.values())

summary = {
    "merged_rows": int(len(merged_df)),
    "cleaned_rows": int(len(cleaned_df)),
    "duplicate_rows_removed": int(duplicate_count),
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "test_rows": int(len(test_df)),
    "missing_values_after_cleaning": int(
        cleaned_df.isna().sum().sum()
    ),
    "cross_split_student_overlaps": overlap_counts,
}

(
    PROJECT_ROOT
    / "artifacts"
    / "preprocessing_summary.json"
).write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Đã lưu preprocessing_summary.json.")

,overlap_rows
train_valid,0
train_test,0
valid_test,0


Đã lưu preprocessing_summary.json.


## Kết luận

Dữ liệu đã được gộp, làm sạch và chia thành 3 tập train, validation và test. Các file cần thiết đã được lưu để sử dụng ở bước tiếp theo. Việc mã hóa và chuẩn hóa sẽ được thực hiện ở notebook sau, chỉ sử dụng tập train để tránh ảnh hưởng đến kết quả đánh giá mô hình.